# 12-1. GTX-A 수요 비중 증가 예측: Ridge Logistic Regression

## 분석 목적

GTX-A 세 역 권역의 250m 격자 126개를 이용하여, GTX 개통 후 한 격자가 **같은 역 권역 전체 광역버스 수요에서 차지하는 비중이 확대되는지** 예측한다.

- `1`: 개통 후 수요 비중 > 개통 전 수요 비중 (**비중 확대**)
- `0`: 개통 후 수요 비중 ≤ 개통 전 수요 비중 (**비중 비확대**)
- 모델 출력 확률은 승차인원 증가율이 아니라 **비중 확대 집단으로 분류될 확률**이다.
- 예측 확률 0.5 이상을 1로 분류한다.

이 분석은 GTX-A에서 학습·검증한 뒤 GTX-B·C의 개통 전 자료에 적용하기 위한 첫 실험이다.

## 종속변수 산정식

격자 $i$가 속한 GTX역 권역을 $s$라고 할 때,

$$
Share^{pre}_{i,s}=\frac{Board^{2024}_{i,s}}{\sum_{j \in s} Board^{2024}_{j,s}}
$$

$$
Share^{post}_{i,s}=\frac{Board^{2025}_{i,s}}{\sum_{j \in s} Board^{2025}_{j,s}}
$$

$$
y_{i,s}=\begin{cases}
1 & \text{if } Share^{post}_{i,s}>Share^{pre}_{i,s}\\
0 & \text{if } Share^{post}_{i,s}\le Share^{pre}_{i,s}
\end{cases}
$$

비중은 반드시 **각 역 권역 내부에서** 계산한다. 따라서 1은 승차인원 자체의 증가를 뜻하지 않는다.

## 모형 구성

두 모형을 동일한 방식으로 검증한다.

1. **기준모형**: 개통 전 수요 비중만 사용
2. **전체모형**: 개통 전 수요 비중 + GTX 접근성 + 인구 + 건축물 연면적 + 경쟁 목적지 비율 + 평균 버스 통행시간 + IC 접근성 + 주거지역 비율

전체모형의 연속형 변수는 표준화하고, 분포가 크게 치우친 인구와 연면적에는 `log1p` 변환을 적용한다. Ridge Logistic Regression은 L2 규제를 사용한다.

In [ ]:
# 프로젝트 루트(UROP) 자동 탐색 — 절대경로 없이 어느 컴퓨터에서도 동작
import sys
from pathlib import Path
UROP_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'config.py').is_file() and (p / 'code').is_dir())
for _d in [UROP_ROOT, *sorted(d for d in (UROP_ROOT / 'code').rglob('*') if d.is_dir() and d.name != '__pycache__')]:
    if str(_d) not in sys.path:
        sys.path.append(str(_d))

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneGroupOut, GridSearchCV
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, roc_auc_score,
    f1_score, brier_score_loss, confusion_matrix
)

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

PROJECT = UROP_ROOT
PROCESSED = PROJECT / 'data' / 'processed' / 'gtx_a'
ID = '셀 ID'

def read_csv(path):
    return pd.read_csv(path, encoding='utf-8-sig', low_memory=False)

MODEL_PATH = PROCESSED / '18_새데이터_회귀분석_격자통합데이터.csv'
EXTRA_PATH = PROCESSED / '18-1_새데이터_추가피처_250m격자.csv'
LAND_USE_PATH = PROCESSED / '19_용도지역_면적비율_250m격자.csv'
FLOOR_PATH = PROCESSED / '24_건축물연면적_2024_250m격자.csv'

model = read_csv(MODEL_PATH)
extra = read_csv(EXTRA_PATH)
land = read_csv(LAND_USE_PATH)
floor = read_csv(FLOOR_PATH)

for frame in [model, extra, land, floor]:
    frame[ID] = pd.to_numeric(frame[ID], errors='coerce').astype('Int64')

df = (
    model
    .merge(extra[[ID, 'bus_time_mean']], on=ID, how='left', validate='one_to_one')
    .merge(land[[ID, '주거지역 비율(%)']], on=ID, how='left', validate='one_to_one')
    .merge(floor[[ID, '2024년 건축물 연면적(천㎡)']], on=ID, how='left', validate='one_to_one')
)

print('통합 데이터:', df.shape)
display(df.head())

## 목표변수 생성 및 확인

역 권역별 2024년·2025년 총승차인원을 분모로 사용해 두 시점의 격자별 수요 비중을 계산한다.

In [ ]:
numeric_columns = [
    'board_2024_all', 'board_2025_all', 'walk_20', 'walk_20_60',
    'pop_20_50', 'competing_share', 'bus_time_mean', 'ic_access',
    '주거지역 비율(%)', '2024년 건축물 연면적(천㎡)'
]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

station_pre_total = df.groupby('nearest_station')['board_2024_all'].transform('sum')
station_post_total = df.groupby('nearest_station')['board_2025_all'].transform('sum')

df['share_pre'] = df['board_2024_all'] / station_pre_total
df['share_post'] = df['board_2025_all'] / station_post_total
df['share_change_pp'] = (df['share_post'] - df['share_pre']) * 100
df['share_increase'] = (df['share_post'] > df['share_pre']).astype(int)

# 인구와 연면적은 우측 꼬리가 길어 log(1+x) 변환
# 같은 변환은 향후 GTX-B·C 입력자료에도 적용해야 한다.
df['log1p_pop_20_50'] = np.log1p(df['pop_20_50'].clip(lower=0))
df['log1p_floor_area'] = np.log1p(df['2024년 건축물 연면적(천㎡)'].clip(lower=0))

class_table = (
    df.groupby(['nearest_station', 'share_increase']).size()
      .unstack(fill_value=0).rename(columns={0: '비확대(0)', 1: '확대(1)'})
)
class_table.loc['전체'] = class_table.sum()
class_table['합계'] = class_table.sum(axis=1)
print('목표변수 분포')
display(class_table)

check = df.groupby('nearest_station').agg(
    격자수=(ID, 'size'),
    개통전_총승차=('board_2024_all', 'sum'),
    개통후_총승차=('board_2025_all', 'sum'),
    개통전_비중합=('share_pre', 'sum'),
    개통후_비중합=('share_post', 'sum')
)
display(check.round(6))

## 독립변수와 분석 표본

`pre_share_pct`는 개통 전 역 권역 내 수요 비중(%)이다. 인구와 연면적은 로그 변환한 값을 투입한다. 모든 입력변수는 각 학습 fold 안에서 결측치를 중앙값으로 대체하고 표준화한다.

In [ ]:
df['pre_share_pct'] = df['share_pre'] * 100

BASELINE_FEATURES = ['pre_share_pct']
FULL_FEATURES = [
    'pre_share_pct',
    'walk_20', 'walk_20_60',
    'log1p_pop_20_50',
    'log1p_floor_area',
    'competing_share',
    'bus_time_mean',
    'ic_access',
    '주거지역 비율(%)'
]
FEATURE_LABELS = {
    'pre_share_pct': '개통 전 역 권역 내 수요 비중(%)',
    'walk_20': 'GTX 보행 20분 이내',
    'walk_20_60': 'GTX 보행 20~60분',
    'log1p_pop_20_50': 'log(1+20~50대 인구)',
    'log1p_floor_area': 'log(1+건축물 연면적 천㎡)',
    'competing_share': 'GTX 경쟁 목적지 비율(%)',
    'bus_time_mean': '평균 버스 통행시간(분)',
    'ic_access': 'IC까지 도로거리(km)',
    '주거지역 비율(%)': '주거지역 비율(%)'
}

required = [ID, 'nearest_station', 'share_increase'] + FULL_FEATURES
analysis_df = df.dropna(subset=[ID, 'nearest_station', 'share_increase']).copy()

summary = pd.DataFrame({
    '변수': [FEATURE_LABELS[x] for x in FULL_FEATURES],
    '결측치': [analysis_df[x].isna().sum() for x in FULL_FEATURES],
    '평균': [analysis_df[x].mean() for x in FULL_FEATURES],
    '표준편차': [analysis_df[x].std() for x in FULL_FEATURES],
    '최솟값': [analysis_df[x].min() for x in FULL_FEATURES],
    '최댓값': [analysis_df[x].max() for x in FULL_FEATURES],
})
print('분석 표본 수:', len(analysis_df))
display(summary.round(3))

## 검증 설계

외부 검증은 `Leave-One-Station-Out` 방식이다.

- 운정중앙 + 킨텍스 학습 → 대곡 검증
- 운정중앙 + 대곡 학습 → 킨텍스 검증
- 킨텍스 + 대곡 학습 → 운정중앙 검증

각 외부 fold에서 Ridge 규제강도 `C`는 검증역을 전혀 사용하지 않고, 두 학습역 사이의 내부 `Leave-One-Station-Out` 검증으로 선택한다. `C`가 작을수록 규제가 강하다.

In [ ]:
def make_pipeline():
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(
            penalty='l2', solver='liblinear', max_iter=5000,
            class_weight=None, random_state=42
        ))
    ])

C_GRID = {'model__C': np.logspace(-3, 3, 13)}
outer_cv = LeaveOneGroupOut()

def nested_station_cv(data, features, model_name):
    X = data[features]
    y = data['share_increase'].astype(int)
    groups = data['nearest_station']
    oof_prob = pd.Series(index=data.index, dtype=float)
    fold_rows = []

    for train_idx, test_idx in outer_cv.split(X, y, groups):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        group_train = groups.iloc[train_idx]
        test_station = groups.iloc[test_idx].iloc[0]

        inner_cv = LeaveOneGroupOut()
        search = GridSearchCV(
            make_pipeline(), C_GRID, scoring='roc_auc',
            cv=inner_cv.split(X_train, y_train, group_train),
            refit=True, n_jobs=-1
        )
        search.fit(X_train, y_train)
        prob = search.predict_proba(X_test)[:, 1]
        pred = (prob >= 0.5).astype(int)
        oof_prob.iloc[test_idx] = prob

        fold_rows.append({
            '모형': model_name,
            '검증역': test_station,
            '학습 수': len(train_idx),
            '검증 수': len(test_idx),
            '검증 1 비율': y_test.mean(),
            '선택 C': search.best_params_['model__C'],
            'Accuracy': accuracy_score(y_test, pred),
            'Balanced Accuracy': balanced_accuracy_score(y_test, pred),
            'ROC-AUC': roc_auc_score(y_test, prob),
            'F1': f1_score(y_test, pred),
            'Brier': brier_score_loss(y_test, prob),
        })

    pred_all = (oof_prob >= 0.5).astype(int)
    overall = {
        '모형': model_name,
        'Accuracy': accuracy_score(y, pred_all),
        'Balanced Accuracy': balanced_accuracy_score(y, pred_all),
        'ROC-AUC': roc_auc_score(y, oof_prob),
        'F1': f1_score(y, pred_all),
        'Brier': brier_score_loss(y, oof_prob),
    }
    return pd.DataFrame(fold_rows), overall, oof_prob

baseline_folds, baseline_overall, baseline_prob = nested_station_cv(
    analysis_df, BASELINE_FEATURES, '기준모형(개통 전 비중)'
)
full_folds, full_overall, full_prob = nested_station_cv(
    analysis_df, FULL_FEATURES, '전체 Ridge Logistic'
)

fold_results = pd.concat([baseline_folds, full_folds], ignore_index=True)
overall_results = pd.DataFrame([baseline_overall, full_overall])

print('===== 역 권역별 외부 검증 =====')
display(fold_results.round(4))
print('===== 세 검증역의 OOF 예측을 합친 전체 성능 =====')
display(overall_results.round(4))

### 지표 해석

- **Accuracy**: 전체 정분류율
- **Balanced Accuracy**: 0과 1의 재현율 평균
- **ROC-AUC**: 임계값과 무관하게 1을 0보다 높게 순위화하는 능력 (`0.5`는 무작위 수준)
- **F1**: 비중 확대(1) 탐지의 정밀도와 재현율 조화평균
- **Brier**: 예측확률과 실제 0·1의 평균제곱오차로, 낮을수록 좋음

세 역에 대한 외부 검증 결과의 편차가 크면 GTX-B·C 일반화 가능성이 낮다는 신호로 해석한다.

In [ ]:
analysis_df['기준모형_확대확률'] = baseline_prob
analysis_df['전체모형_확대확률'] = full_prob
analysis_df['전체모형_예측'] = (full_prob >= 0.5).astype(int)

cm = confusion_matrix(analysis_df['share_increase'], analysis_df['전체모형_예측'])
cm_table = pd.DataFrame(
    cm,
    index=['실제 비확대(0)', '실제 확대(1)'],
    columns=['예측 비확대(0)', '예측 확대(1)']
)
print('===== 전체모형 OOF 혼동행렬 =====')
display(cm_table)

prediction_table = analysis_df[[
    ID, 'nearest_station', 'board_2024_all', 'board_2025_all',
    'share_pre', 'share_post', 'share_change_pp', 'share_increase',
    '전체모형_확대확률', '전체모형_예측'
]].copy()
prediction_table['share_pre'] *= 100
prediction_table['share_post'] *= 100
prediction_table = prediction_table.rename(columns={
    'nearest_station': 'GTX-A역 권역',
    'board_2024_all': '2024년 승차인원',
    'board_2025_all': '2025년 승차인원',
    'share_pre': '개통 전 수요 비중(%)',
    'share_post': '개통 후 수요 비중(%)',
    'share_change_pp': '수요 비중 변화(%p)',
    'share_increase': '실제 비중 확대 여부',
    '전체모형_확대확률': '비중 확대 예측확률',
    '전체모형_예측': '예측 비중 확대 여부'
})
display(prediction_table.sort_values('비중 확대 예측확률', ascending=False).head(15).round(4))

## 전체 GTX-A 자료로 최종모형 적합

GTX-B·C에 적용할 최종모형의 `C`는 GTX-A 세 역을 그룹으로 둔 `Leave-One-Station-Out` 교차검증으로 선택한다. 아래 계수는 표준화된 입력 기준이며, 절댓값이 클수록 모형의 분류에 더 크게 작용한다. 다만 표본과 역 권역 수가 적으므로 인과효과나 확정적인 영향요인으로 해석하지 않는다.

In [ ]:
X_all = analysis_df[FULL_FEATURES]
y_all = analysis_df['share_increase'].astype(int)
groups_all = analysis_df['nearest_station']

final_search = GridSearchCV(
    make_pipeline(), C_GRID, scoring='roc_auc',
    cv=LeaveOneGroupOut().split(X_all, y_all, groups_all),
    refit=True, n_jobs=-1
)
final_search.fit(X_all, y_all)
final_model = final_search.best_estimator_

coef = final_model.named_steps['model'].coef_[0]
coef_table = pd.DataFrame({
    '변수': [FEATURE_LABELS[x] for x in FULL_FEATURES],
    '표준화 계수': coef,
    '승산비 exp(계수)': np.exp(coef)
}).sort_values('표준화 계수', ascending=False)

print('최종 선택 C:', final_search.best_params_['model__C'])
print('GTX-A 역 권역별 CV 평균 ROC-AUC:', round(final_search.best_score_, 4))
display(coef_table.round(4))

fig, ax = plt.subplots(figsize=(8, 5.5))
plot_coef = coef_table.sort_values('표준화 계수')
colors = np.where(plot_coef['표준화 계수'] >= 0, '#d95f02', '#1b9e77')
ax.barh(plot_coef['변수'], plot_coef['표준화 계수'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('표준화 계수')
ax.set_title('전체 GTX-A 자료로 적합한 Ridge Logistic 계수')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## 판단 기준과 주의사항

1. 전체모형이 기준모형보다 역 권역별 검증에서 일관되게 좋아지는지 확인한다.
2. 전체 성능만 보지 않고 대곡·킨텍스·운정중앙 각 fold의 성능 편차를 확인한다.
3. `비중 확대 확률`은 승차인원 증가율이나 비중 증가 폭이 아니다.
4. 한 권역 안에서 비중의 합은 항상 100%이므로 격자 관측치가 완전히 독립적이지 않다.
5. 경쟁 목적지 비율은 GTX-B·C에서도 노선별 경쟁 목적지를 같은 원칙으로 정의할 수 있어야 한다.
6. GTX-A 역이 3개뿐이므로 이 결과는 예비모형 검증으로 해석한다. B·C 적용 전 변수 범위와 지역 특성의 차이를 반드시 점검한다.

## 최초 실행 결과 기록 (2026-10-11)

역 권역별 외부검증의 OOF 예측 결과는 다음과 같다.

| 모형 | Accuracy | Balanced Accuracy | ROC-AUC | F1 | Brier |
|---|---:|---:|---:|---:|---:|
| 기준모형(개통 전 비중) | 0.357 | 0.355 | 0.427 | 0.400 | 0.250 |
| 전체 Ridge Logistic | 0.524 | 0.522 | 0.523 | 0.552 | 0.254 |

전체 Ridge Logistic의 검증역별 결과:

| 검증역 | Accuracy | Balanced Accuracy | ROC-AUC |
|---|---:|---:|---:|
| 대곡 | 0.514 | 0.513 | 0.541 |
| 운정중앙 | 0.524 | 0.511 | 0.496 |
| 킨텍스 | 0.539 | 0.554 | 0.571 |

### 현재 판단

- 전체모형은 개통 전 비중만 사용한 기준모형보다는 나았지만, ROC-AUC와 Balanced Accuracy가 약 0.52로 **현재 예측력은 무작위 수준에 가깝다**.
- 따라서 이 결과만으로 GTX-B·C 예측을 진행하거나 개별 변수의 효과를 확정해서는 안 된다.
- 다음 실험에서는 `수요 비중 변화량(%p)` 자체를 이용한 연속형 Ridge, 매우 작은 기초수요 격자에 대한 민감도 분석, 경쟁 목적지 변수의 B·C 공통 정의 가능성을 차례로 점검할 필요가 있다.
- 표본이 126개여도 계산은 가능하지만, 독립된 역 권역이 3개뿐이라는 점이 지역 일반화 성능의 핵심 제약이다.

※ 위 수치는 노트북 코드와 동일한 환경에서 별도 실행하여 확인한 값이다. VS Code에서 전체 실행하면 각 표와 혼동행렬, 계수 그림이 셀 출력으로 생성된다.

## 추가 실험: 40명 이상 격자 + 보행시간 연속형

광역버스 1대의 대략적인 수용 규모를 40명으로 보고, 개통 전 승차인원이 40명 이상인 격자만 학습한다. 수요 비중 자체는 기존과 같이 역 권역의 **전체 126개 격자 수요 합계**를 분모로 계산한 뒤 표본만 제한한다.

GTX 접근성은 `20분 이내`, `20~60분` 더미 대신 `walk_min`을 연속형으로 투입한다. 나머지 학습·검증 절차는 앞의 전체모형과 같다.

In [ ]:
THRESHOLD_2024 = 40
sample_40 = analysis_df[analysis_df['board_2024_all'] >= THRESHOLD_2024].copy()

FEATURES_40_CONTINUOUS_WALK = [
    'pre_share_pct',
    'walk_min',
    'log1p_pop_20_50',
    'log1p_floor_area',
    'competing_share',
    'bus_time_mean',
    'ic_access',
    '주거지역 비율(%)'
]
FEATURE_LABELS['walk_min'] = 'GTX역까지 보행시간(분)'

class_40 = (
    sample_40.groupby(['nearest_station', 'share_increase']).size()
             .unstack(fill_value=0)
             .rename(columns={0: '비확대(0)', 1: '확대(1)'})
)
class_40.loc['전체'] = class_40.sum()
class_40['합계'] = class_40.sum(axis=1)
print('2024년 승차인원 40명 이상 표본의 목표변수 분포')
display(class_40)

folds_40, overall_40, probability_40 = nested_station_cv(
    sample_40,
    FEATURES_40_CONTINUOUS_WALK,
    '40명 이상 + 보행시간 연속형'
)
print('===== 역 권역별 외부 검증 =====')
display(folds_40.round(4))
print('===== OOF 전체 성능 =====')
display(pd.DataFrame([overall_40]).round(4))

### 40명 기준 실험 결과

| 표본 | 비확대(0) | 확대(1) | 합계 |
|---|---:|---:|---:|
| 대곡 | 11 | 7 | 18 |
| 운정중앙 | 21 | 13 | 34 |
| 킨텍스 | 11 | 10 | 21 |
| 전체 | 43 | 30 | 73 |

| 지표 | 결과 |
|---|---:|
| Accuracy | 0.534 |
| Balanced Accuracy | 0.504 |
| ROC-AUC | 0.478 |
| F1 | 0.370 |
| Brier | 0.267 |

40명 기준은 미세 수요 격자의 불안정성을 줄이는 명확한 실무적 근거가 있지만, 표본이 126개에서 73개로 감소했다. 역 권역별 ROC-AUC도 기존 전체모형의 0.523에서 0.478로 낮아져 현재 자료에서는 예측력 향상으로 이어지지 않았다.

따라서 40명 기준 결과는 중요한 **민감도 분석**으로 제시하되, 이를 성능 향상을 이유로 주모형으로 선택하기는 어렵다. 보행시간 연속형의 효과만 분리해서 판단하려면 다음으로 전체 126개를 유지한 상태에서 `walk_min`만 바꾼 모형을 비교해야 한다.

## 분리 실험: 전체 126개 + 보행시간 연속형

40명 필터 효과와 보행시간 변수 변경 효과를 구분하기 위해, 전체 126개 격자를 유지하고 접근성 변수만 `walk_min` 연속형으로 바꾼다.

In [ ]:
folds_walk_cont, overall_walk_cont, probability_walk_cont = nested_station_cv(
    analysis_df,
    FEATURES_40_CONTINUOUS_WALK,
    '전체 126개 + 보행시간 연속형'
)
print('===== 역 권역별 외부 검증 =====')
display(folds_walk_cont.round(4))
print('===== OOF 전체 성능 =====')
display(pd.DataFrame([overall_walk_cont]).round(4))

model_comparison_all = pd.DataFrame([
    full_overall,
    overall_walk_cont,
    overall_40
])
print('===== 세 모형 비교 =====')
display(model_comparison_all.round(4))

### 세 모형 비교 결과

| 모형 | 표본 수 | Balanced Accuracy | ROC-AUC | Brier |
|---|---:|---:|---:|---:|
| 전체 126개 + 보행구간 더미 | 126 | 0.522 | 0.523 | 0.254 |
| 전체 126개 + 보행시간 연속형 | 126 | 0.515 | **0.561** | **0.249** |
| 40명 이상 + 보행시간 연속형 | 73 | 0.504 | 0.478 | 0.267 |

보행시간을 연속형으로 바꾸면 ROC-AUC와 확률오차(Brier)는 개선됐다. 반면 40명 기준으로 표본을 제한하면 53개 격자가 제외되어 성능이 악화됐다. 현재 결과에서는 **전체 126개 + 보행시간 연속형**을 우선모형으로 두고, **40명 이상 결과는 실질 수요 격자에 대한 민감도 분석**으로 제시하는 구성이 적절하다.

## 추가 실험: 전체 126개 + 40명 기준 학습 가중치

격자를 제외하지 않고 전체 126개를 유지하면서 소규모 수요 격자의 학습 영향만 줄인다.

$$
w_i=\min\left(\frac{Board^{2024}_i}{40},1\right)
$$

예를 들어 1명은 0.025, 10명은 0.25, 20명은 0.5, 40명 이상은 1.0의 학습 가중치를 갖는다. 보행 접근성은 `walk_min` 연속형을 사용한다.

모형 간 공정한 비교를 위해 일반 지표를 먼저 보고, 40명 기준의 목적을 반영한 수요가중 지표를 추가로 제시한다.

In [ ]:
def nested_station_cv_weighted(data, features, model_name):
    X = data[features]
    y = data['share_increase'].astype(int)
    groups = data['nearest_station']
    weights = np.minimum(data['board_2024_all'] / 40.0, 1.0)
    oof_prob = pd.Series(index=data.index, dtype=float)
    fold_rows = []

    for train_idx, test_idx in LeaveOneGroupOut().split(X, y, groups):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        group_train = groups.iloc[train_idx]
        w_train, w_test = weights.iloc[train_idx], weights.iloc[test_idx]
        test_station = groups.iloc[test_idx].iloc[0]

        search = GridSearchCV(
            make_pipeline(), C_GRID, scoring='roc_auc',
            cv=LeaveOneGroupOut().split(X_train, y_train, group_train),
            refit=True, n_jobs=-1
        )
        search.fit(
            X_train, y_train,
            model__sample_weight=w_train.to_numpy()
        )
        prob = search.predict_proba(X_test)[:, 1]
        pred = (prob >= 0.5).astype(int)
        oof_prob.iloc[test_idx] = prob

        fold_rows.append({
            '모형': model_name,
            '검증역': test_station,
            '학습 수': len(train_idx),
            '검증 수': len(test_idx),
            '선택 C': search.best_params_['model__C'],
            'Accuracy': accuracy_score(y_test, pred),
            'Balanced Accuracy': balanced_accuracy_score(y_test, pred),
            'ROC-AUC': roc_auc_score(y_test, prob),
            'F1': f1_score(y_test, pred),
            'Brier': brier_score_loss(y_test, prob),
            '가중 Accuracy': accuracy_score(y_test, pred, sample_weight=w_test),
            '가중 Balanced Accuracy': balanced_accuracy_score(y_test, pred, sample_weight=w_test),
            '가중 ROC-AUC': roc_auc_score(y_test, prob, sample_weight=w_test),
            '가중 F1': f1_score(y_test, pred, sample_weight=w_test),
            '가중 Brier': brier_score_loss(y_test, prob, sample_weight=w_test),
        })

    pred_all = (oof_prob >= 0.5).astype(int)
    overall = {
        '모형': model_name,
        'Accuracy': accuracy_score(y, pred_all),
        'Balanced Accuracy': balanced_accuracy_score(y, pred_all),
        'ROC-AUC': roc_auc_score(y, oof_prob),
        'F1': f1_score(y, pred_all),
        'Brier': brier_score_loss(y, oof_prob),
        '가중 Accuracy': accuracy_score(y, pred_all, sample_weight=weights),
        '가중 Balanced Accuracy': balanced_accuracy_score(y, pred_all, sample_weight=weights),
        '가중 ROC-AUC': roc_auc_score(y, oof_prob, sample_weight=weights),
        '가중 F1': f1_score(y, pred_all, sample_weight=weights),
        '가중 Brier': brier_score_loss(y, oof_prob, sample_weight=weights),
    }
    return pd.DataFrame(fold_rows), overall, oof_prob, weights

weighted_folds, weighted_overall, weighted_prob, demand_weights = nested_station_cv_weighted(
    analysis_df,
    FEATURES_40_CONTINUOUS_WALK,
    '전체 126개 + 40명 기준 가중치'
)
print('===== 가중 Ridge Logistic: 역 권역별 외부 검증 =====')
display(weighted_folds.round(4))
print('===== 가중 Ridge Logistic: OOF 전체 성능 =====')
display(pd.DataFrame([weighted_overall]).round(4))
print('===== 학습 가중치 분포 =====')
display(demand_weights.describe().to_frame('학습 가중치').round(4))

### 가중모형 결과

| 평가 방식 | Accuracy | Balanced Accuracy | ROC-AUC | F1 | Brier |
|---|---:|---:|---:|---:|---:|
| 일반 지표 | 0.508 | 0.508 | 0.502 | 0.516 | 0.265 |
| 40명 기준 수요가중 지표 | 0.483 | 0.480 | 0.473 | 0.442 | 0.266 |

가중치 없는 `전체 126개 + 보행시간 연속형`의 ROC-AUC 0.561보다 낮다. 따라서 현재 결과에서는 40명 기준 가중모형을 최종모형으로 선택할 근거가 없다.

다만 이 결과는 중요한 진단이다. 소규모 격자의 우연한 변동만이 낮은 예측력의 주된 원인은 아니며, 큰 수요 격자에서도 세 GTX-A 역 사이의 관계가 일관되지 않음을 보여준다. 현재 우선모형은 전체 126개를 사용하는 보행시간 연속형 모형으로 유지하고, 40명 필터와 가중모형은 민감도 분석으로 제시한다.

## 변수 제거 진단(Leave-One-Feature-Out)

전체 126개·보행시간 연속형 모형에서 독립변수를 하나씩 제외하고, 매번 Ridge 규제강도를 다시 선택하여 역 권역별 교차검증을 수행한다.

어떤 변수를 제외했을 때 성능이 반복적으로 개선되면 해당 변수가 다른 역 권역 예측에 잡음을 추가할 가능성이 있다. 다만 같은 외부검증 결과를 보고 변수를 고르면 선택 편향이 생기므로, 아래 결과는 최종 변수선정이 아니라 **후보 진단**으로 사용한다.

In [ ]:
def leave_one_feature_out(data, features):
    rows = []
    full_folds, full_result, _ = nested_station_cv(
        data, features, '전체 변수'
    )
    rows.append({
        '모형': '전체 변수',
        'Accuracy': full_result['Accuracy'],
        'Balanced Accuracy': full_result['Balanced Accuracy'],
        'ROC-AUC': full_result['ROC-AUC'],
        'ROC-AUC 변화': 0.0,
    })

    for excluded in features:
        selected = [x for x in features if x != excluded]
        _, result, _ = nested_station_cv(
            data, selected, f'{FEATURE_LABELS.get(excluded, excluded)} 제외'
        )
        rows.append({
            '모형': f'{FEATURE_LABELS.get(excluded, excluded)} 제외',
            'Accuracy': result['Accuracy'],
            'Balanced Accuracy': result['Balanced Accuracy'],
            'ROC-AUC': result['ROC-AUC'],
            'ROC-AUC 변화': result['ROC-AUC'] - full_result['ROC-AUC'],
        })
    return pd.DataFrame(rows).sort_values('ROC-AUC', ascending=False)

ablation_results = leave_one_feature_out(
    analysis_df, FEATURES_40_CONTINUOUS_WALK
)
display(ablation_results.round(4))

### 변수 제거 진단 결과

| 제외 변수 | Accuracy | Balanced Accuracy | ROC-AUC | 전체모형 대비 AUC 변화 |
|---|---:|---:|---:|---:|
| 보행시간 | 0.563 | 0.559 | **0.575** | +0.013 |
| IC까지 도로거리 | **0.579** | **0.576** | 0.573 | +0.011 |
| 평균 버스 통행시간 | 0.540 | 0.537 | 0.568 | +0.007 |
| 개통 전 수요 비중 | 0.532 | 0.532 | 0.568 | +0.006 |
| 제외 없음 | 0.516 | 0.515 | 0.561 | 기준 |
| 건축물 연면적 | 0.540 | 0.538 | 0.555 | -0.007 |
| 주거지역 비율 | 0.524 | 0.527 | 0.540 | -0.022 |
| 경쟁 목적지 비율 | 0.492 | 0.490 | 0.537 | -0.024 |
| 20~50대 인구 | 0.492 | 0.492 | 0.530 | -0.031 |

현재 자료에서는 보행시간, IC 거리, 평균 버스 통행시간, 개통 전 수요 비중이 결합모형의 역 권역 외부 예측을 약간 방해하는 후보로 나타났다. 반면 경쟁 목적지 비율과 20~50대 인구를 제외하면 AUC가 비교적 크게 감소했다.

다만 최고 AUC도 0.575로 여전히 낮고, 세 개 역만으로 여러 변수 조합을 비교했으므로 우연히 개선됐을 가능성이 있다. 특히 보행시간은 연구상 핵심 변수이므로 점수만 보고 바로 제거하기보다 역별 관계 방향, 비선형성, 경쟁 목적지 비율과의 상호작용을 확인해야 한다.